<center><h1>Loke_Eliza_HW3</h1></center>
<br>
<br>

Name: Eliza Loke
<br>
Github Username: eliza-data
<br>
USC ID: 9840375742

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [23]:
import pandas as pd
import numpy as np
from pathlib import Path

Get the AReM Data Set

In [24]:
# Directory setup for data files 
NOTEBOOK_DIR = Path('.').resolve()

if NOTEBOOK_DIR.name.strip().lower() in ['notebook', 'notebooks']:
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

# Step up to the project root directory, then point to AReM
DATA_DIR = PROJECT_ROOT / 'AReM'

# Defensive check
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Data directory not found at: {DATA_DIR}\n"
        "Check 'AReM' folder located in the project root folder."
    )

print(f"Project Root: {PROJECT_ROOT}")
print(f"Dataset Directory: {DATA_DIR}")

Project Root: /Users/eliza/Desktop/DSCI 552 Machine Learning /F26/Homework Folder/Homework 3, 4 Data
Dataset Directory: /Users/eliza/Desktop/DSCI 552 Machine Learning /F26/Homework Folder/Homework 3, 4 Data/AReM


### (b) Test and Train Data

In [25]:
# function to split data into train or test based on the folder name and file name
def get_split(folder_name, file_name):
    # Extract the dataset number from file name, e.g., 'dataset1.csv' -> 1
    dataset_num = int(''.join(filter(str.isdigit, file_name)))
    
    if folder_name in ['bending1', 'bending2']:
        return 'test' if dataset_num in [1, 2] else 'train'
    else:
        return 'test' if dataset_num in [1, 2, 3] else 'train'
    
    
### checks get_split function and prints summary statements ###
# Collections to hold file paths
train_files = []
test_files = []

# Loop over all activity folders inside AReM
for folder_path in sorted(DATA_DIR.iterdir()):
    if folder_path.is_dir() and not folder_path.name.startswith('.'):
        folder_name = folder_path.name
        
        # Iterate through every CSV file in the activity folder
        for file_path in sorted(folder_path.glob('*.csv')):
            file_name = file_path.name
            split_type = get_split(folder_name, file_name)
            
            if split_type == 'test':
                test_files.append((folder_name, file_name))
            else:
                train_files.append((folder_name, file_name))

# Print summary counts
print(f"Total Train Instances: {len(train_files)}")
print(f"Total Test Instances:  {len(test_files)}")
print(f"Total Combined:        {len(train_files) + len(test_files)}")
print("-" * 50)

# Detailed printout of Test Files per folder
print("TEST FILES PER FOLDER:")
for folder, file in test_files:
    print(f"  [{folder}] -> {file}")

Total Train Instances: 69
Total Test Instances:  19
Total Combined:        88
--------------------------------------------------
TEST FILES PER FOLDER:
  [bending1] -> dataset1.csv
  [bending1] -> dataset2.csv
  [bending2] -> dataset1.csv
  [bending2] -> dataset2.csv
  [cycling] -> dataset1.csv
  [cycling] -> dataset2.csv
  [cycling] -> dataset3.csv
  [lying] -> dataset1.csv
  [lying] -> dataset2.csv
  [lying] -> dataset3.csv
  [sitting] -> dataset1.csv
  [sitting] -> dataset2.csv
  [sitting] -> dataset3.csv
  [standing] -> dataset1.csv
  [standing] -> dataset2.csv
  [standing] -> dataset3.csv
  [walking] -> dataset1.csv
  [walking] -> dataset2.csv
  [walking] -> dataset3.csv


### (c) Feature Extraction

#### i. Research

Time Series Classification (Types of time-domain features)

Basic Statistical Features
1. Minimum
2. Maximum
3. Mean
4. Median
5. Standard Deviation 
6. First Quartile 
7. Thrid Quartile 

Distribution Features
1. Skewness
2. Kurtosis
3. Interquartile Range (IQR) 
4. Peak-to-Peak Range

Energy & Magnitude Features
1. Root Mean Square (RMS)
2. Signal Energy
3. Mean Absolute Deviation (MAD)
4. Signal Energy 
5. Integral of Absolute Value (IAV)

Temporal Dynamics Features
1. Zero-Crossing Rate (ZCR)
2. Mean Absolute Difference

#### ii. Extraction

In [26]:
def extract_file_features(file_path, activity_name, split_name, instance_id):
    # Try reading with skiprows=4 first due to csv file fomrat
    df = pd.read_csv(file_path, skiprows=4, on_bad_lines='skip', engine='python')
    
    # Fallback check if headers are offset differently
    if not any(col in df.columns for col in ['avg_rss12', 'var_rss12']):
        df = pd.read_csv(file_path, skiprows=3, on_bad_lines='skip', engine='python')

    sensor_cols = [col for col in df.columns if str(col).strip() in ['avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23', 'var_rss23']]
    
    if not sensor_cols:
        sensor_cols = list(df.columns[:6])
        
    row_data = {
        'Instance': instance_id,
        'File_Name': file_path.name,
        'Activity': activity_name,
        'Split': split_name
    }
    
    for i, col in enumerate(sensor_cols, start=1):
        series = pd.to_numeric(df[col], errors='coerce').dropna() # Drop any NaN/empty rows safely
        
        # Check if the series is empty to avoid warnings 
        if len(series) == 0:
            row_data[f'min_{i}'] = np.nan
            row_data[f'max_{i}'] = np.nan
            row_data[f'mean_{i}'] = np.nan
            row_data[f'median_{i}'] = np.nan
            row_data[f'std_{i}'] = np.nan
            row_data[f'1st_quart_{i}'] = np.nan
            row_data[f'3rd_quart_{i}'] = np.nan
        else:
            row_data[f'min_{i}'] = series.min()
            row_data[f'max_{i}'] = series.max()
            row_data[f'mean_{i}'] = series.mean()
            row_data[f'median_{i}'] = series.median()
            row_data[f'std_{i}'] = series.std()
            row_data[f'1st_quart_{i}'] = series.quantile(0.25)
            row_data[f'3rd_quart_{i}'] = series.quantile(0.75)
        
    return row_data

# -- Main --- Process All 88 Instances 
feature_records = []
instance_counter = 1

for folder_path in sorted(DATA_DIR.iterdir()):
    if folder_path.is_dir() and not folder_path.name.startswith('.'):
        folder_name = folder_path.name
        
        for file_path in sorted(folder_path.glob('*.csv')):
            file_name = file_path.name
            split_type = get_split(folder_name, file_name)
            
            features = extract_file_features(file_path, folder_name, split_type, instance_counter)
            feature_records.append(features)
            instance_counter += 1

# Display the Final Table 
feature_matrix = pd.DataFrame(feature_records)

print(f"Feature Matrix Shape: {feature_matrix.shape}")

# Displays table right in notebook
display(feature_matrix)

Feature Matrix Shape: (88, 46)


,Instance,File_Name,Activity,Split,min_1,max_1,mean_1,median_1,std_1,1st_quart_1,...,std_5,1st_quart_5,3rd_quart_5,min_6,max_6,mean_6,median_6,std_6,1st_quart_6,3rd_quart_6
0,1,dataset1.csv,bending1,test,37.25,45.00,40.624792,40.500,1.476967,39.25,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.430,0.582915,0.00,1.3000
1,2,dataset2.csv,bending1,test,38.00,45.67,42.812812,42.500,1.435550,42.00,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.430,0.601010,0.00,1.3000
2,3,dataset3.csv,bending1,train,35.00,47.40,43.954500,44.330,1.558835,43.00,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.430,0.513506,0.00,0.9400
3,4,dataset4.csv,bending1,train,33.00,47.75,42.179812,43.500,3.670666,39.15,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.500,0.524317,0.00,1.0000
4,5,dataset5.csv,bending1,train,33.00,45.75,41.678063,41.750,2.243490,41.33,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.430,0.389164,0.00,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,84,dataset5.csv,walking,train,20.75,46.25,34.763333,35.290,4.742208,31.67,...,3.174681,14.2500,18.33,0.0,9.39,3.288271,3.270,1.647528,2.05,4.3050
84,85,dataset6.csv,walking,train,21.50,51.00,34.935812,35.500,4.645944,32.00,...,3.192058,14.2375,18.25,0.0,10.21,3.280021,3.015,1.700918,2.12,4.5000
85,86,dataset7.csv,walking,train,18.33,47.67,34.333042,34.750,4.948770,31.25,...,3.000493,13.7500,18.00,0.0,8.01,3.261583,2.980,1.617290,2.05,4.3200
86,87,dataset8.csv,walking,train,18.33,45.75,34.599875,35.125,4.731790,31.50,...,2.905688,14.0000,18.25,0.0,8.86,3.289542,3.015,1.680170,2.12,4.2600


#### iii. Standard Deviation

In [27]:
# Separate feature columns from metadata 
metadata_cols = ['Instance', 'File_Name', 'Activity', 'Split']
feature_cols = [col for col in feature_matrix.columns if col not in metadata_cols]

# Set up bootstrap parameters
n_bootstrap = 1000       
confidence_level = 0.90  
alpha = 1 - confidence_level

np.random.seed(42)       
bootstrap_results = []

# Loop through each of the 42 features
for col in feature_cols:
    data = feature_matrix[col].dropna().values
    
    # Calculate the sample standard deviation for feature columns
    sample_std = np.std(data, ddof=1)
    
    # Run bootstrap resampling
    bootstrap_stds = []
    for _ in range(n_bootstrap):
        resample = np.random.choice(data, size=len(data), replace=True)
        bootstrap_stds.append(np.std(resample, ddof=1))
        
    # Calculate 90% CI percentiles (5th and 95th percentiles)
    ci_lower = np.percentile(bootstrap_stds, (alpha / 2) * 100)
    ci_upper = np.percentile(bootstrap_stds, (1 - alpha / 2) * 100)
    
    # Save results
    bootstrap_results.append({
        'Feature': col,
        'Sample_Std': sample_std,
        '90%_CI_Lower': ci_lower,
        '90%_CI_Upper': ci_upper
    })

# Convert results into a summary DataFrame and display it
ci_summary_df = pd.DataFrame(bootstrap_results)

print("Bootstrap Standard Deviation Confidence Intervals (90%):")
display(ci_summary_df.head(10)) 

Bootstrap Standard Deviation Confidence Intervals (90%):


,Feature,Sample_Std,90%_CI_Lower,90%_CI_Upper
0,min_1,9.624011,8.326581,10.737514
1,max_1,4.207745,3.143285,5.123189
2,mean_1,5.276413,4.667255,5.847675
3,median_1,5.386624,4.710145,5.969270
4,std_1,1.771313,1.568442,1.946131
5,1st_quart_1,6.128143,5.516528,6.639048
6,3rd_quart_1,5.031028,4.166218,5.707824
7,min_2,0.000000,0.000000,0.000000
8,max_2,5.059656,4.614432,5.398190
9,mean_2,1.577941,1.416915,1.712585


#### iv. Select Features

1. min_1
2. max_1	
3. mean_1

## 2. ISLR 3.7.4

### (a) Linear Train

It'd be expected the training RSS for cubic regression to be lower than the training RSS for linear regression. Cubic regression is known for it's flexibility due to greater degrees of freedom, which allows model predictions has less errors and training predictions are more aligned with the training dataset. Therefore, the RSS for cubic regression is lower than linear regression. 

### (b) Linear Test

It'd be expected the test RSS for linear regression to be lower than the training RSS for cubic regression. Due to the flexibility and greater degrees of freedom, it is highly proability to result in overfitting. The test data is unseen, making the cubic regression to "chase" the noise, while linear regression offers a generalized model that's more ideal for test data.  Therefore, the RSS for linear regression is lower than cubic regression. 

### (c) Not Linear Train

It'd be expected the training RSS for cubic regression to be lower than the training RSS for linear regression. Due to it's flexibility, as mentioned in previous question it continues to "chase" the noise. For this reason, it cubic regression will be lower than linear regression considering the true relationship between X and Y (linear or non-linear) will not alter the behavior of a cubic regression for training RSS as it will "follow" the noise. 

### (d) Not Linear Testing

In a test RSS, there is not enough informaton to determine whether RSS for cubic regression to be lower, equal, or higher than RSS for linear regression. It is known the severity of the non-linear relationshpi between X and Y. If it's minimal then linear regression is lower due to its low variance nature. If there's high curviture, cubic regression will capture the pattern, making it lower. Overall it can not be deterined based on the given information. 

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 